In [1]:
from curl_cffi import requests as curl_requests
import pandas as pd
import json
import time
import random
import os

In [2]:
# ============ CHANGE THESE AS NEEDED ============

CITY = "gurgaon"
PROPERTY_TYPE = "independent_house"  # Options: "flats", "independent_house", "builder_floor"

MAX_PAGES_PER_LOCALITY = 70  # 99acres caps at ~70 pages per URL

# Throttling
MIN_DELAY = 1      # min seconds between page requests
MAX_DELAY = 3      # max seconds between page requests
BATCH_PAUSE = 20   # extra pause every BATCH_SIZE pages
BATCH_SIZE = 10

# ============ AUTO-CONFIGURED (don't change) ============

# URL prefix for each property type
URL_PREFIX_MAP = {
    "flats": "flats-in",
    "independent_house": "independent-house-in",
    "builder_floor": "independent-builder-floors-in",
}

FILE_PREFIX_MAP = {
    "flats": "flats",
    "independent_house": "independent_house",
    "builder_floor": "builder_floor",
}

# Filter: only keep properties matching this PROPERTY_TYPE value from JSON
PROPERTY_TYPE_FILTER = {
    "flats": "Residential Apartment",
    "independent_house": "Independent House/Villa",
    "builder_floor": "Builder Floor",
}

URL_PREFIX = URL_PREFIX_MAP[PROPERTY_TYPE]
CITY_BASE_URL = f"https://www.99acres.com/{URL_PREFIX}-{CITY}-ffid"
OUTPUT_DIR = "../../data/web_scraping"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"{FILE_PREFIX_MAP[PROPERTY_TYPE]}_{CITY}.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Property Type: {PROPERTY_TYPE}")
print(f"Filter: PROPERTY_TYPE == '{PROPERTY_TYPE_FILTER[PROPERTY_TYPE]}'")
print(f"City: {CITY}")
print(f"City URL: {CITY_BASE_URL}")
print(f"Output: {OUTPUT_FILE}")

Property Type: independent_house
Filter: PROPERTY_TYPE == 'Independent House/Villa'
City: gurgaon
City URL: https://www.99acres.com/independent-house-in-gurgaon-ffid
Output: ../../data/web_scraping\independent_house_gurgaon.csv


In [3]:
def load_scraped_ids(output_file):
    if os.path.exists(output_file):
        df = pd.read_csv(output_file)
        ids = set(df["property_id"].dropna().astype(str).tolist())
        print(f"Resuming: {len(ids)} properties already scraped.")
        return ids
    print("Fresh start.")
    return set()

def save_page_data(records, output_file):
    if not records:
        return
    df = pd.DataFrame(records)
    write_header = not os.path.exists(output_file)
    df.to_csv(output_file, mode="a", header=write_header, index=False)

# --- Progress tracking for locality-level resume ---
PROGRESS_FILE = os.path.join(OUTPUT_DIR, f".progress_{FILE_PREFIX_MAP[PROPERTY_TYPE]}_{CITY}.json")

def load_progress():
    """Load set of completed locality slugs from progress file."""
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            progress = json.load(f)
        return set(progress.get("completed_localities", []))
    return set()

def save_progress(completed_localities):
    """Save completed locality slugs to progress file."""
    with open(PROGRESS_FILE, "w") as f:
        json.dump({"completed_localities": sorted(completed_localities)}, f)

scraped_ids = load_scraped_ids(OUTPUT_FILE)
completed_localities = load_progress()
print(f"Completed localities: {len(completed_localities)}")

Resuming: 2054 properties already scraped.
Completed localities: 105


In [4]:
FACING_MAP = {
    "1": "East", "2": "West", "3": "North", "4": "South",
    "5": "North-East", "6": "North-West", "7": "South-East", "8": "South-West",
}

FURNISH_MAP = {
    "1": "Furnished", "2": "Semi-Furnished", "3": "Unfurnished", "4": "Unfurnished",
}

OVERLOOKING_MAP = {
    "1": "Park/Garden", "2": "Main Road", "3": "Club",
    "4": "Pool", "5": "Others", "7": "Sea facing",
}

def map_codes(code_str, mapping):
    """Convert comma-separated numeric codes to readable names using a mapping dict."""
    if not code_str:
        return ""
    codes = str(code_str).split(",")
    names = [mapping.get(c.strip(), c.strip()) for c in codes]
    return ", ".join(names)

def extract_property(p):
    """Convert a raw JSON property dict into the output format."""
    landmarks = p.get("LANDMARK_DETAILS") or []
    nearby = [lm.get("name", "") for lm in landmarks if lm.get("name")]

    area_val = p.get("AREA", "")
    area_type = (p.get("AREA_TYPE") or "").replace("_", " ").title()
    area_with_type = f"{area_val} {area_type}".strip() if area_val else ""

    floor_num = p.get("FLOOR_NUM", "")
    total_floor = p.get("TOTAL_FLOOR", "")
    floor_info = f"{floor_num} of {total_floor} Floors" if floor_num and total_floor else str(floor_num)

    facing_code = str(p.get("FACING", ""))
    facing = FACING_MAP.get(facing_code, facing_code)

    furnish_code = str(p.get("FURNISH", ""))
    furnish = FURNISH_MAP.get(furnish_code, furnish_code)

    overlooking = map_codes(p.get("OVERLOOKING", ""), OVERLOOKING_MAP)

    features = p.get("USP_V2_FH_REMOVED", "")

    map_details = p.get("MAP_DETAILS") or {}
    latitude = map_details.get("LATITUDE", "")
    longitude = map_details.get("LONGITUDE", "")

    parking = p.get("RESERVED_PARKING", "")

    return {
        "property_name": p.get("PROP_HEADING", ""),
        "link": "https://www.99acres.com" + p.get("PD_URL", ""),
        "society": p.get("SOCIETY_NAME") or p.get("BUILDING_NAME", ""),
        "price": p.get("FORMATTED_PRICE") or p.get("PRICE", ""),
        "area": p.get("LOCALIZED_PRICE_SQFT_TEXT", ""),
        "areaWithType": area_with_type,
        "carpetArea": p.get("CARPET_AREA", ""),
        "pricePerSqft": p.get("LOCALIZED_PRICE_SQFT", ""),
        "bedRoom": p.get("BEDROOM_NUM", ""),
        "bathroom": p.get("BATHROOM_NUM", ""),
        "balcony": p.get("BALCONY_NUM", ""),
        "address": p.get("LOCALITY", ""),
        "floorNum": floor_info,
        "facing": facing,
        "overlooking": overlooking,
        "agePossession": p.get("AGE", ""),
        "cornerProperty": p.get("CORNER_PROPERTY", ""),
        "furnishing": furnish,
        "parking": parking,
        "nearbyLocations": nearby if nearby else "",
        "description": p.get("DESCRIPTION", ""),
        "features": features,
        "latitude": latitude,
        "longitude": longitude,
        "property_id": p.get("PROP_ID", ""),
    }

print("Extractor ready.")

Extractor ready.


In [5]:
def discover_localities(city_url):
    """Fetch the city-wide listing page and extract locality facets."""
    print(f"Discovering localities from: {city_url}")
    resp = curl_requests.get(city_url, impersonate="chrome120", timeout=30)

    if resp.status_code != 200:
        raise Exception(f"Failed to fetch city page: HTTP {resp.status_code}")

    html = resp.text
    marker = "window.__initialData__="
    idx = html.find(marker)
    if idx == -1:
        raise Exception("No __initialData__ found on city page")

    decoder = json.JSONDecoder()
    data, _ = decoder.raw_decode(html[idx + len(marker):])
    localities = data["srp"]["pageData"]["facets"]["LOCALITY_ID"]

    results = []
    for loc in localities:
        slug = (loc["label"].lower()
                .strip()
                .replace(" ", "-")
                .replace("--", "-")
                .strip("-"))
        # Remove trailing city name to avoid duplication in URL
        if slug.endswith(f"-{CITY}"):
            slug = slug[:-len(f"-{CITY}")]
        results.append({
            "id": loc["id"],
            "label": loc["label"],
            "count": loc["count"],
            "slug": slug,
        })

    results.sort(key=lambda x: x["count"], reverse=True)
    return results

localities = discover_localities(CITY_BASE_URL)
total_listed = sum(loc["count"] for loc in localities)
pending = [loc for loc in localities if loc["slug"] not in completed_localities]

print(f"\nFound {len(localities)} localities with ~{total_listed} total listings")
print(f"Already completed: {len(completed_localities)}")
print(f"Remaining: {len(pending)}")
print(f"\nTop 10 localities by count:")
for loc in localities[:10]:
    status = "✅" if loc["slug"] in completed_localities else "⏳"
    print(f"  {status} {loc['label']}: {loc['count']} listings (slug: {loc['slug']})")

Discovering localities from: https://www.99acres.com/independent-house-in-gurgaon-ffid

Found 99 localities with ~1953 total listings
Already completed: 105
Remaining: 0

Top 10 localities by count:
  ✅ Sector 109 Gurgaon: 114 listings (slug: sector-109)
  ✅ DLF Phase 2: 78 listings (slug: dlf-phase-2)
  ✅ Sector 46 Gurgaon: 70 listings (slug: sector-46)
  ✅ Palam Vihar: 61 listings (slug: palam-vihar)
  ✅ Sector 48 Gurgaon: 52 listings (slug: sector-48)
  ✅ Sector 66 Gurgaon: 51 listings (slug: sector-66)
  ✅ DLF Phase 1: 50 listings (slug: dlf-phase-1)
  ✅ Nirvana Country: 50 listings (slug: nirvana-country)
  ✅ Sector 4 Gurgaon: 48 listings (slug: sector-4)
  ✅ Sector 82 Gurgaon: 46 listings (slug: sector-82)


In [6]:
MAX_CONSECUTIVE_ERRORS = 5
expected_type = PROPERTY_TYPE_FILTER[PROPERTY_TYPE]
total_new = 0
rate_limited = False
total_requests = 0

pending = [loc for loc in localities if loc["slug"] not in completed_localities]

for loc_idx, locality in enumerate(pending):
    slug = locality["slug"]
    label = locality["label"]
    base_url = f"https://www.99acres.com/{URL_PREFIX}-{slug}-{CITY}-ffid"

    print(f"\n{'='*60}")
    print(f"LOCALITY {loc_idx+1}/{len(pending)}: {label} (~{locality['count']} listings)")
    print(f"URL: {base_url}")
    print(f"{'='*60}")

    locality_new = 0
    consecutive_errors = 0

    for page in range(1, MAX_PAGES_PER_LOCALITY + 1):
        url = f"{base_url}-page-{page}" if page > 1 else base_url

        try:
            resp = curl_requests.get(url, impersonate="chrome120", timeout=30)
            total_requests += 1

            # HTTP error
            if resp.status_code != 200:
                if resp.status_code in (410, 404):
                    # 410/404 = no more pages for this locality
                    if page == 1:
                        print(f"  Page {page}: HTTP {resp.status_code} — URL may be invalid, skipping locality")
                    else:
                        print(f"  Page {page}: HTTP {resp.status_code} — end of pages")
                    break
                consecutive_errors += 1
                print(f"  Page {page}: HTTP {resp.status_code} (error {consecutive_errors}/{MAX_CONSECUTIVE_ERRORS})")
                if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                    print(f"  ⛔ Rate limited! Stopping.")
                    rate_limited = True
                    break
                continue

            html = resp.text
            marker = "window.__initialData__="
            idx = html.find(marker)

            if idx == -1:
                consecutive_errors += 1
                print(f"  Page {page}: No JSON (error {consecutive_errors}/{MAX_CONSECUTIVE_ERRORS})")
                if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                    rate_limited = True
                    break
                continue

            decoder = json.JSONDecoder()
            data, _ = decoder.raw_decode(html[idx + len(marker):])

            if "srp" not in data:
                consecutive_errors += 1
                print(f"  Page {page}: No 'srp' key (error {consecutive_errors}/{MAX_CONSECUTIVE_ERRORS})")
                if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                    rate_limited = True
                    break
                continue

            properties = data["srp"]["pageData"]["properties"]
            individual = [
                p for p in properties
                if p.get("entityType") is None and p.get("PROPERTY_TYPE") == expected_type
            ]

            # No individual listings = end of useful pages
            if not individual and page > 1:
                print(f"  Page {page}: 0 listings — end of pages")
                break

            new_records = []
            for p in individual:
                pid = str(p.get("PROP_ID", ""))
                if pid and pid not in scraped_ids:
                    new_records.append(extract_property(p))
                    scraped_ids.add(pid)

            save_page_data(new_records, OUTPUT_FILE)
            locality_new += len(new_records)
            total_new += len(new_records)
            consecutive_errors = 0

            print(f"  Page {page}: {len(individual)} listings, {len(new_records)} new — locality: {locality_new}, total: {total_new}")

        except Exception as e:
            consecutive_errors += 1
            print(f"  Page {page}: Error — {e} (error {consecutive_errors}/{MAX_CONSECUTIVE_ERRORS})")
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                rate_limited = True
                break
            continue

        # Throttle
        if total_requests % BATCH_SIZE == 0:
            time.sleep(BATCH_PAUSE)
        else:
            time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

    # If not rate limited, mark locality as completed
    if not rate_limited:
        completed_localities.add(slug)
        save_progress(completed_localities)
        print(f"  ✅ {label} done — {locality_new} new properties")
    else:
        print(f"\n⛔ Stopped due to rate limiting after {total_new} new properties.")
        print(f"✅ Progress saved ({len(completed_localities)} localities done). Just re-run to resume.")
        break

    # Extra pause between localities
    if loc_idx < len(pending) - 1:
        pause = random.uniform(5, 10)
        print(f"  Switching locality (pause {pause:.0f}s)...")
        time.sleep(pause)

print(f"\n{'='*60}")
print(f"SUMMARY: {total_new} new properties scraped across {len(completed_localities)} localities")
print(f"Total unique properties: {len(scraped_ids)}")
print(f"Output: {OUTPUT_FILE}")
print(f"{'='*60}")


SUMMARY: 0 new properties scraped across 105 localities
Total unique properties: 2054
Output: ../../data/web_scraping\independent_house_gurgaon.csv


### Feature Enrichment (Detail Page Scraping)

For independent houses, `USP_V2_FH_REMOVED` is empty in the listing page JSON. Features are only available on each property's **detail page**. This cell visits each property link and extracts features from the detail page JSON.

~2034 properties × ~2 sec each = ~1 hour. Saves after every batch so re-runs skip completed ones.

In [7]:
# DEBUG: Test one detail page to extract features using HTML parsing
from bs4 import BeautifulSoup

df_test = pd.read_csv(OUTPUT_FILE)
test_url = df_test.iloc[0]['link']
print(f"Testing: {test_url}\n")

resp = curl_requests.get(test_url, impersonate="chrome120", timeout=30)
print(f"Status: {resp.status_code}")

soup = BeautifulSoup(resp.text, 'html.parser')

# Same approach as old Selenium scraper: div[data-label='FACILITIES'] > ul#features li div
container = soup.select_one("div[data-label='FACILITIES']")
if container:
    items = container.select("ul#features li div")
    features = [item.text.strip() for item in items if item.text.strip()]
    print(f"\nFeatures found ({len(features)}): {features}")
else:
    print("\nNo FACILITIES container found. Trying alternative selectors...")
    # Try other selectors
    for selector in ["ul#features li div", "div.features_tags div", "div.amenitiesWrap li"]:
        items = soup.select(selector)
        if items:
            features = [item.text.strip() for item in items if item.text.strip()]
            print(f"  Selector '{selector}': {features[:10]}")
            break
    else:
        print("  No features found with any selector")

Testing: https://www.99acres.com/5-bhk-bedroom-independent-house-villa-for-sale-in-sobha-international-city-phase-4-sector-109-gurgaon-3600-sqft-r1-spid-L87277236

Status: 200

Features found (7): ['Vaastu Compliant', 'Private Garden / Terrace', 'Maintenance Staff', 'Water Storage', 'Park', 'Visitor Parking', 'Rain Water Harvesting']


In [8]:
from bs4 import BeautifulSoup
from curl_cffi.requests import Session

def fetch_features_from_detail(session, url):
    """Fetch detail page and extract features using HTML parsing."""
    resp = session.get(url, timeout=30)
    if resp.status_code != 200:
        return None, resp.status_code

    soup = BeautifulSoup(resp.text, 'html.parser')
    container = soup.select_one("div[data-label='FACILITIES']")
    if container:
        items = container.select("ul#features li div")
        features = [item.text.strip() for item in items if item.text.strip()]
        return features if features else None, 200
    return None, 200

# --- Load data and find what needs enrichment ---
df = pd.read_csv(OUTPUT_FILE)
df["features"] = df["features"].astype(str).replace("nan", "")
print(f"Total properties: {len(df)}")

needs_features = df[df['features'].isin(['', 'nan'])].index.tolist()
print(f"Properties needing feature enrichment: {len(needs_features)}")

if not needs_features:
    print("All properties already have features!")

# --- Slow-and-steady strategy ---
# One consistent session, generous delays, long cooldowns.
# Mimics a real human browsing — slow but doesn't get blocked.

BASE_DELAY = 5            # seconds between requests (human-like pace)
BATCH_SIZE = 20           # pause after every N requests
BATCH_PAUSE = 30          # seconds to pause between batches
COOLDOWN_WAIT = 300       # 5 min cooldown on rate limit
MAX_COOLDOWNS = 10        # stop after this many
BATCH_SAVE = 50

session = Session(impersonate="chrome120")
enriched = 0
no_features = 0
errors = 0
consecutive_errors = 0
cooldown_count = 0

print(f"Strategy: {BASE_DELAY}s between requests, {BATCH_PAUSE}s pause every {BATCH_SIZE} reqs, {COOLDOWN_WAIT}s cooldown")
print(f"Estimated time: ~{len(needs_features) * (BASE_DELAY + 1) / 60:.0f} min + pauses\n")

for i, idx in enumerate(needs_features):
    link = df.at[idx, 'link']

    try:
        features, status = fetch_features_from_detail(session, link)

        if features:
            df.at[idx, 'features'] = str(features)
            enriched += 1
            consecutive_errors = 0
        elif status == 200:
            df.at[idx, 'features'] = "none"
            no_features += 1
            consecutive_errors = 0
        else:
            errors += 1
            consecutive_errors += 1

    except Exception as e:
        errors += 1
        consecutive_errors += 1

    # Rate limit → long cooldown with fresh session
    if consecutive_errors >= 5:
        cooldown_count += 1
        if cooldown_count > MAX_COOLDOWNS:
            print(f"\n⛔ Too many cooldowns ({cooldown_count}). Saving and stopping.")
            break

        df.to_csv(OUTPUT_FILE, index=False)
        print(f"\n  ⏸️  Rate limited — cooldown #{cooldown_count}: waiting {COOLDOWN_WAIT}s...")
        time.sleep(COOLDOWN_WAIT)

        session.close()
        session = Session(impersonate="chrome120")
        consecutive_errors = 0
        print(f"  ▶️  Resumed")

    # Progress log
    if (i + 1) % 10 == 0:
        pct = (i + 1) / len(needs_features) * 100
        print(f"  [{pct:5.1f}%] {i+1}/{len(needs_features)} — enriched: {enriched}, none: {no_features}, errors: {errors}")

    # Save periodically
    if (i + 1) % BATCH_SAVE == 0:
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"  💾 Saved at {i+1}")

    # Throttle: longer pause every BATCH_SIZE, else normal delay
    if (i + 1) % BATCH_SIZE == 0:
        print(f"  ⏳ Batch pause {BATCH_PAUSE}s...")
        time.sleep(BATCH_PAUSE)
    else:
        time.sleep(random.uniform(BASE_DELAY * 0.8, BASE_DELAY * 1.2))

session.close()
df.to_csv(OUTPUT_FILE, index=False)

still_empty = df['features'].isin(['', 'nan']).sum()
print(f"\n{'='*50}")
print(f"Done! Enriched: {enriched} | No features: {no_features} | Errors: {errors}")
print(f"Cooldowns used: {cooldown_count}")
print(f"Features still empty: {still_empty}")

Total properties: 2054
Properties needing feature enrichment: 599
Strategy: 5s between requests, 30s pause every 20 reqs, 300s cooldown
Estimated time: ~60 min + pauses

  [  1.7%] 10/599 — enriched: 1, none: 9, errors: 0
  [  3.3%] 20/599 — enriched: 4, none: 16, errors: 0
  ⏳ Batch pause 30s...
  [  5.0%] 30/599 — enriched: 11, none: 19, errors: 0
  [  6.7%] 40/599 — enriched: 17, none: 23, errors: 0
  ⏳ Batch pause 30s...
  [  8.3%] 50/599 — enriched: 23, none: 27, errors: 0
  💾 Saved at 50
  [ 10.0%] 60/599 — enriched: 31, none: 29, errors: 0
  ⏳ Batch pause 30s...
  [ 11.7%] 70/599 — enriched: 40, none: 30, errors: 0
  [ 13.4%] 80/599 — enriched: 44, none: 36, errors: 0
  ⏳ Batch pause 30s...
  [ 15.0%] 90/599 — enriched: 52, none: 38, errors: 0
  [ 16.7%] 100/599 — enriched: 59, none: 41, errors: 0
  💾 Saved at 100
  ⏳ Batch pause 30s...
  [ 18.4%] 110/599 — enriched: 59, none: 51, errors: 0
  [ 20.0%] 120/599 — enriched: 59, none: 61, errors: 0
  ⏳ Batch pause 30s...
  [ 21.7%] 